<a href="https://colab.research.google.com/github/luzcielponce-create/MS-E-226-Project-/blob/main/(REVISED)prepping_models_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prediction section

**Import Packages & Set Seed**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (log_loss, accuracy_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 226
N_FOLDS = 5
THRESHOLD = 0.5

In [ ]:
df = pd.read_csv("explore_80.csv")

# Replace empty cells with missing values.
df = df.replace(r"^\s*$", np.nan, regex=True)

y = df["Netflix"].astype(int)  # What we're predicting
X = df.drop(columns="Netflix").copy()  # Potential predictors

genre_columns = []
for column in X.columns:
    if column.startswith("genre_"):
        genre_columns.append(column)

# Create column indicating whether genres are missing: 1 means missing & 0 means known
X["genres_missing"] = X["genres"].isna().astype(int)

# For titles with unknown genres, set genre indicators to 0
# Use genres_missing column to identify these titles
X.loc[X["genres_missing"] == 1, genre_columns] = 0

# Keep missing age classifications as "unrated".
X["Age"] = X["Age"].fillna("unrated")

print("Total titles:", len(df))
print("Titles on Netflix:", y.sum())
print("Netflix percentage:", round(y.mean() * 100, 2), "%")
missing_runtime = X["runtime_minutes"].isna().sum() #Count missing runtimes
print("Missing runtimes:", missing_runtime)

Total titles: 11906
Titles on Netflix: 4491
Netflix percentage: 37.72 %
Missing runtimes: 3083


**Folds**

In [ ]:
#CREATE THE FIVE FOLDS ONCE
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,     #Randomly mix titles before dividing them
    random_state=226  #Use same random split every time
)

# Generate training and validation indices for all 5 folds
cv_splits = list(cv.split(X, y))

# Look at number of titles in each fold
for fold in range(5):
    # Get training and validation indices for this fold
    train_index, validation_index = cv_splits[fold]

    # Print how many titles belong to each group
    print("Fold:", fold + 1)
    print("Training titles:", len(train_index))
    print("Validation titles:", len(validation_index))
    print()


Fold: 1
Training titles: 9524
Validation titles: 2382

Fold: 2
Training titles: 9525
Validation titles: 2381

Fold: 3
Training titles: 9525
Validation titles: 2381

Fold: 4
Training titles: 9525
Validation titles: 2381

Fold: 5
Training titles: 9525
Validation titles: 2381



**Preprocessing**

In [ ]:
# Create custom tool to fill missing runtimes
class RuntimeMedianImputer(BaseEstimator, TransformerMixin):
    # FIT: Learn replacement values from training data
    def fit(self, X, y=None):
        # Make sure runtime values are numeric
        runtimes = pd.to_numeric(
            X["runtime_minutes"], errors="coerce"
        )

        # Calculate median runtime for movies and TV shows
        self.medians_ = runtimes.groupby(
            X["content_type"]
        ).median()

        # Calculate overall median as backup
        self.fallback_ = runtimes.median()

        # Stop if no runtime values are available
        if pd.isna(self.fallback_):
            raise ValueError("No observed training runtimes.")

        # Return fitted imputer
        return self

    # TRANSFORM: Fill missing values using learned medians
    def transform(self, X):
        # Copy data to avoid changing the original
        out = X.copy()

        runtimes = pd.to_numeric(
            out["runtime_minutes"], errors="coerce"
        )

        # Record which runtimes were originally missing (1 = missing & 0 = observed)
        out["runtime_missing"] = runtimes.isna().astype(int)

        # Find correct median for each content type
        replacements = out["content_type"].map(self.medians_)

        # Use overall median if a group's median is unavailable
        replacements = replacements.fillna(self.fallback_)

        # Fill missing runtimes with replacement values
        out["runtime_minutes"] = runtimes.fillna(replacements)

        # Return the updated dataset
        return out


# Function that builds preprocessing pipeline
def make_preprocessor(scale_numeric=True):
    # Choose whether to standardize runtime.
    if scale_numeric:
        numeric_transform = StandardScaler()
    else:
        numeric_transform = "passthrough"

    # List binary predictors we want to use
    binary_columns = [
        "Type",
        "runtime_missing",
        "genres_missing"
    ] + genre_columns

    # How to process each group of predictors
    encoding = ColumnTransformer(
        transformers=[
            # Standardize runtime if requested.
            (
                "numeric",
                numeric_transform,
                ["runtime_minutes"]
            ),

            # Convert Age into numerical category indicators.
            (
                "age",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["Age"]
            ),

            # Keep binary predictors unchanged.
            (
                "binary",
                "passthrough",
                binary_columns
            )
        ],

        # Remove every column not explicitly selected above.
        remainder="drop"
    )

    # First fill runtime, then encode and select features
    return Pipeline([
        ("runtime", RuntimeMedianImputer()),
        ("encoding", encoding)
    ])


**Run Preprocessing across all five folds**

In [ ]:
# Store imputation results from each fold
imputation_rows = []

# Repeat preprocessing for all 5 folds
for fold in range(5):

    # Get row indices for this fold
    train_index, validation_index = cv_splits[fold]

    # Select training and validation data
    X_train = X.iloc[train_index].copy()
    X_validation = X.iloc[validation_index].copy()

    # PREPROCESS THE DATA!!
    # Create preprocessing pipeline
    preprocessor = make_preprocessor(scale_numeric=True)

    # Learn medians and other preprocessing settings using
    # only training data, then transform it
    prepared_train = preprocessor.fit_transform(X_train)

    # Apply SAME learned preprocessing to validation data
    prepared_validation = preprocessor.transform(X_validation)

    # Make sure there are no missing or infinite numbers
    assert np.isfinite(prepared_train).all()
    assert np.isfinite(prepared_validation).all()

    # Get runtime imputer from our pipeline
    imputer = preprocessor.named_steps["runtime"]

    # Store information about the runtime imputation
    imputation_rows.append({
        "Fold": fold + 1,
        "Movie median": imputer.medians_["Movie"],
        "TV median": imputer.medians_["TV Show"],
        "Validation runtimes filled":
            X_validation["runtime_minutes"].isna().sum()
    })

# Convert list of results into a DataFrame
imputation_results = pd.DataFrame(imputation_rows)

# Display runtime imputation results for each fold
print("RUNTIME IMPUTATION BY FOLD")
print(imputation_results)

RUNTIME IMPUTATION BY FOLD
   Fold  Movie median  TV median  Validation runtimes filled
0     1          95.0       40.0                         610
1     2          95.0       40.0                         587
2     3          95.0       40.0                         612
3     4          95.0       40.0                         631
4     5          96.0       41.0                         643


**SUMMARY: Calculate average model performance**

In [ ]:
print("\nSUMMARY")

# Calculate average validation log loss across all 5 folds
mean_log_loss = baseline_results["Log loss"].mean()
print("Mean validation log loss:", round(mean_log_loss, 6))

# Calculate how much log loss varies across the 5 folds
std_log_loss = baseline_results["Log loss"].std(ddof=1)
print("Standard deviation of log loss:", round(std_log_loss, 6))

# Calculate average misclassification rate as a percentage
mean_error = baseline_results["Misclassification rate"].mean()
print("Mean misclassification rate:", round(mean_error * 100, 2), "%")

# Calculate average recall as a percentage
mean_recall = baseline_results["Recall"].mean()
print("Mean recall:", round(mean_recall * 100, 2), "%")

# Check whether precision is missing for every fold
if baseline_results["Precision"].isna().all():
    print("Precision is undefined: no positive predictions were made")

else:
    # Display the precision results for all 5 folds.
    print("Precision by fold:", baseline_results["Precision"].tolist())


SUMMARY
Mean validation log loss: 0.662679
Standard deviation of log loss: 5.9e-05
Mean misclassification rate: 37.72 %
Mean recall: 0.0 %
Precision is undefined: no positive predictions were made


**Record proposed Tuning Settings (records part b plan, but does not fit 3 predictive models yet)**

In [ ]:
TUNING_PLAN = {
    # MODEL 1: Logistic Regression
    "logistic_regression": {

        # Use L1 regularization (aka lasso)
        # This can shrink some predictor coefficients to zero
        "regularization": "L1 (lasso)",
        # Test four different regularization strengths
        "C": [0.01, 0.1, 1, 10],
        # Make randomized operations reproducible
        "random_state": SEED
    },

    # MODEL 2: Classification Tree
    "classification_tree": {
        # Test three maximum tree depths
        "max_depth": [3, 5, 8],
        # Test three minimum leaf sizes
        "min_samples_leaf": [25, 50, 100],
        # Make randomized operations reproducible
        "random_state": SEED
    },

    # MODEL 3: Random Forest
    "random_forest": {
        # Set random seed
        "random_state": SEED

        # Keep all other settings at their defaults
    }
}


**Evaluate Baseline Model**

QUESTION: How accurately can we predict Netflix availability without using any information about individual titles?

ANSWER: Use proportion of Netflix titles in each training fold as our baseline probability, then evaluate that prediction on the corresponding validation fold.

In [ ]:
# Baseline uses only Netflix outcomes, not title characteristics
results = []

# Repeat baseline evaluation for all 5 folds
for fold in range(5):
    # Get the training and validation indices
    train_index, validation_index = cv_splits[fold]

    # Select the actual Netflix outcomes for each group
    y_train = y.iloc[train_index]
    y_validation = y.iloc[validation_index]

    # CALCULATE BASELINE PROBABILITY!
    # Calculate percentage of training titles on Netflix (because y contains 0s and 1s, its mean is the proportion of 1s)
    baseline_probability = y_train.mean()

    # Give every validation title the same probability
    probabilities = np.full(
        len(y_validation),
        baseline_probability
    )

    # MAKE BINARY PREDICTIONS!
    # Predict Netflix = 1 if probability is at least 0.5, otherwise, predict Netflix = 0
    predictions = (probabilities >= 0.5).astype(int)

    # CALCULATE EVALUATION METRICS!
    # Log loss measures quality of predicted probabilities
    fold_log_loss = log_loss(
        y_validation, probabilities, labels=[0, 1]
    )

    # Misclassification rate= percentage predicted incorrectly
    fold_error = 100 * (
        1 - accuracy_score(y_validation, predictions)
    )

    # Recall= how many available titles we identified
    fold_recall = 100 * recall_score(
        y_validation, predictions, zero_division=0
    )

    # Precision is undefined if we make no positive predictions
    if predictions.sum() == 0:
        fold_precision = np.nan
    else:
        fold_precision = 100 * precision_score(
            y_validation, predictions
        )

    # SAVE RESULTS FOR THIS FOLD!
    results.append({
        "Fold": fold + 1,
        "Training Netflix %": 100 * baseline_probability,
        "Log loss": fold_log_loss,
        "Misclassification %": fold_error,
        "Precision %": fold_precision,
        "Recall %": fold_recall
    })

# Convert results into a table
baseline_results = pd.DataFrame(results)

# Display results for each fold
print("RESULTS FOR EACH FOLD")
print(baseline_results.round(6))

# Calculate average performance across the five folds
print("\nAVERAGE PERFORMANCE ACROSS FOLDS")

metrics = [
    "Log loss",
    "Misclassification %",
    "Precision %",
    "Recall %"
]

print(baseline_results[metrics].mean().round(6))
print("\nNaN precision means undefined, not zero.")

RESULTS FOR EACH FOLD
   Fold  Training Netflix %  Log loss  Misclassification %  Precision %  \
0     1           37.715246  0.662784            37.741394          NaN   
1     2           37.721785  0.662653            37.715246          NaN   
2     3           37.721785  0.662653            37.715246          NaN   
3     4           37.721785  0.662653            37.715246          NaN   
4     5           37.721785  0.662653            37.715246          NaN   

   Recall %  
0       0.0  
1       0.0  
2       0.0  
3       0.0  
4       0.0  

AVERAGE PERFORMANCE ACROSS FOLDS
Log loss                0.662679
Misclassification %    37.720475
Precision %                  NaN
Recall %                0.000000
dtype: float64

NaN precision means undefined, not zero.


**Define 2 Feature Sets!**

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Three models we plan to compare
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

genre_columns = []
for column in X.columns:
    if column.startswith("genre_"):
        genre_columns.append(column)


#SHARED BINARY PREDICTORS!
# These predictors will be used in BOTH feature sets
shared_binary = [
    "Type",             #1 = TV Show, 0 = Movie
    "runtime_missing",  #1 = runtime was originally missing
    "genres_missing"    #1 = genre information is missing
]

# Add 28 individual genre indicators
shared_binary = shared_binary + genre_columns


# TWO FEATURE SETS HERE!
feature_sets = {
    # Set 1: Simpler group of predictors.
    "Set 1": {
        # Numeric predictors that may need scaling
        "numeric": ["runtime_minutes"],
        # Binary predictors that are already 0 or 1
        "binary": shared_binary,
        # Categorical predictors that need one-hot encoding
        "categorical": ["Age"]
    },

    # Set 2: Set 1 plus three additional numeric predictors
    "Set 2": {
        "numeric": [
            "runtime_minutes",
            "imdb_rating",
            "rt_score",
            "imdb_num_votes"
        ],
        "binary": shared_binary,
        "categorical": ["Age"]
    }
}

print("Number of genre indicators:", len(genre_columns))

Number of genre indicators: 28


**Prepare Variables**

Clean and convert Netflix predictors into a format that our models can understand.

In [ ]:
# Create custom preprocessing tool
class PrepareTitleFeatures(BaseEstimator, TransformerMixin):

    # Save list of genre columns
    def __init__(self, genre_columns):
        self.genre_columns = genre_columns

    # No values need to be learned from training data here
    def fit(self, X, y=None):
        return self

    # Clean and convert the predictor columns
    def transform(self, X):
        # Copy so we don't change the original data
        out = X.copy()

        # Convert empty cells and spaces into missing values
        out = out.replace(r"^\s*$", np.nan, regex=True)

        # Replace missing age ratings with "unrated"
        out["Age"] = out["Age"].fillna("unrated")

        # Create an indicator for missing genres (1 = missing, 0 = known)
        out["genres_missing"] = out["genres"].isna().astype(int)

        # For titles with missing genres, set their individual genre indicators to 0
        # The genres_missing flag distinguishes these placeholders from known genre absences
        out.loc[
            out["genres_missing"] == 1,
            self.genre_columns
        ] = 0

        # Convert Rotten Tomatoes scores into numbers (ex. "66/100" becomes 66)
        rt_text = out["Rotten Tomatoes"].astype("string")
        rt_text = rt_text.str.replace("/100", "", regex=False)

        out["rt_score"] = pd.to_numeric(
            rt_text, errors="coerce"
        )

        # Convert IMDb ratings into numbers
        out["imdb_rating"] = pd.to_numeric(
            out["imdb_rating"], errors="coerce"
        )

        # Convert IMDb vote counts into numbers
        out["imdb_num_votes"] = pd.to_numeric(
            out["imdb_num_votes"], errors="coerce"
        )

        # Return cleaned dataset
        return out

**Defining our three predictive algorithms and their tuning settings.**

In [ ]:
# Dictionary containing our three algorithms
models = {
    # MODEL 1
    "Logistic regression": LogisticRegression(
        penalty="l1",        # Use L1 (lasso) regularization
        solver="liblinear",  # Algorithm that fits logistic regression
        max_iter=2000,       # Maximum number of iterations allowed
        random_state=SEED    # Set random seed for reproducibility
    ),
    # MODEL 2: repeatedly divides titles into smaller groups based on their characteristics
    "Classification tree": DecisionTreeClassifier(
        max_depth=5,         # Maximum depth of the tree
        min_samples_leaf=50, # Minimum number of titles in each leaf
        random_state=SEED
    ),
    # MODEL 3: train many decision trees and combine their predictions
    "Random forest": RandomForestClassifier(
        random_state=SEED    # Keep other parameters at defaults
    )
}

# WHICH SETTINGS WE WANT TO TEST!
param_grids = {
    # Test 4 different regularization strengths
    "Logistic regression": {
        "model__C": [0.01, 0.1, 1, 10]
    },
    # Test combinations of tree depth and minimum leaf size
    "Classification tree": {
        "model__max_depth": [3, 5, 8],
        "model__min_samples_leaf": [25, 50, 100]
    },
    # No parameter tuning for random forest
    "Random forest": {}
}

**Combining preprocessing with predictive models.**

In [ ]:
# Build a pipeline for ONE feature set and ONE predictive model
def build_model_pipeline(set_name, model_name):
    # Get predictor columns belonging to this feature set
    features = feature_sets[set_name]

    # PREPARE NUMERIC PREDICTORS!
    # Standarize variables
    if model_name == "Logistic regression":
        numeric_scaling = StandardScaler()
    else:
        numeric_scaling = "passthrough"

    # Fill missing numeric values using training medians
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", numeric_scaling)
    ])

    # HOW TO PROCESS EACH FEATURE TYPE!
    encoding = ColumnTransformer([
        # Numeric predictors: impute and optionally scale
        (
            "numeric",
            numeric_steps,
            features["numeric"]
        ),
        # Age ratings: convert categories into 0/1 columns
        (
            "age",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            features["categorical"]
        ),
        # Binary predictors: keep existing 0/1 values
        (
            "binary",
            "passthrough",
            features["binary"]
        )
    ], remainder="drop")

    # COMBINE CLEANING, IMPUTATION, AND MODELING!
    return Pipeline([
        # Clean and convert predictor columns
        ("prepare", PrepareTitleFeatures(genre_columns)),

        # Fill missing runtimes using movie/TV training medians
        ("runtime", RuntimeMedianImputer()),

        # Apply transformations to the selected predictors
        ("encoding", encoding),

        # Add the chosen predictive algorithm.

        ("model", clone(models[model_name])) #clone() makes a fresh, unfitted copy of the model
    ])


# CREATE ALL SIX PIPELINES!
pipelines = {}
for set_name in feature_sets:
    # Go through each of our three models
    for model_name in models:
        # Build pipeline for this combination
        pipeline = build_model_pipeline(set_name, model_name)
        # Store it using the feature set and model name as keys
        pipelines[(set_name, model_name)] = pipeline

# DISPLAY PIPELINES!
for set_name, model_name in pipelines:
    print(set_name, ":", model_name)

Set 1 : Logistic regression
Set 1 : Classification tree
Set 1 : Random forest
Set 2 : Logistic regression
Set 2 : Classification tree
Set 2 : Random forest


**DEFINE EVALUATION METRICS**

log loss selects the settings

other metrics use fixed threshold 0.5

In [ ]:
from sklearn.model_selection import ParameterGrid
THRESHOLD = 0.5

# CALCULATE MODEL PERFORMANCE FOR ONE FOLD!
def score_predictions(y_true, probabilities):

    # Convert predicted probabilities into 0/1 classifications
    # 1 = predicted available on Netflix
    # 0 = predicted unavailable
    predictions = (probabilities >= THRESHOLD).astype(int)

    # Calculate log loss using probabilities
    fold_log_loss = log_loss(
        y_true, probabilities, labels=[0, 1]
    )

    # Calculate percentage of incorrect classifications
    fold_error = 100 * (
        1 - accuracy_score(y_true, predictions)
    )

    # Calculate recall as a percentage
    fold_recall = 100 * recall_score(
        y_true, predictions, zero_division=0
    )

    # Precision is undefined if we predict no positive titles
    if predictions.sum() == 0:
        fold_precision = np.nan
    else:
        fold_precision = 100 * precision_score(
            y_true, predictions
        )

    # Return all four metrics together in a dictionary
    return {
        "Log loss": fold_log_loss,
        "Error (%)": fold_error,
        "Precision (%)": fold_precision,
        "Recall (%)": fold_recall
    }


# SUMMARIZE RESULTS ACROSS FIVE FOLDS!
def summarize_folds(records):

    # Convert list of fold results into a DataFrame
    scores = pd.DataFrame(records)

    # Calculate average log loss
    mean_log_loss = scores["Log loss"].mean()

    # Calculate how much log loss varies across folds
    sd_log_loss = scores["Log loss"].std(ddof=1)

    # Calculate average misclassification percentage
    mean_error = scores["Error (%)"].mean()

    # Calculate average recall percentage
    mean_recall = scores["Recall (%)"].mean()

    # Only calculate average precision if every fold has a defined precision value
    if scores["Precision (%)"].notna().all():
        mean_precision = scores["Precision (%)"].mean()
    else:
        mean_precision = np.nan

    # Return summary results
    return {
        "Log loss": mean_log_loss,
        "Log loss SD": sd_log_loss,
        "Error (%)": mean_error,
        "Precision (%)": mean_precision,
        "Recall (%)": mean_recall
    }


# DISPLAY TUNING SETTINGS!
def settings_text(parameters):

    # Store readable parameter names
    descriptions = []
    # Go through each tuning parameter and its chosen value
    for key, value in parameters.items():

        # Remove "model__" from the parameter name
        name = key.replace("model__", "")

        # Combine name and value into readable text
        descriptions.append(f"{name}={value}")

    # If there are no tuning parameters, report defaults
    if len(descriptions) == 0:
        return "Defaults; random_state=226"

    # Join parameter descriptions with semicolons.
    return "; ".join(descriptions)


**Train and evaluate every combination**

main training block; everything starts with a fresh pipeline so imputation, encoding, and scaling are learned within that fold's training portion


In [ ]:
candidate_rows = []  # Average results for each model configuration
fold_rows = []       # Results for each individual fold

# COUNT HOW MANY MODELS WE WILL FIT!
total_candidates = 0

# Go through each feature set and algorithm combination
for set_name, model_name in pipelines:

    # Get all tuning combinations for this algorithm
    grid = list(ParameterGrid(param_grids[model_name]))

    # Add number of configurations to our total
    total_candidates = total_candidates + len(grid)

print("Total configurations:", total_candidates)
print("Total model fits:", total_candidates * len(cv_splits))


# GO THROUGH OUR SIX PIPELINES!
for (set_name, model_name), template in pipelines.items():

    # Retrieve the tuning configurations for this algorithm
    grid = list(ParameterGrid(param_grids[model_name]))

    print("\nFeature set:", set_name)
    print("Algorithm:", model_name)

    # TEST EACH HYPERPARAMETER CONFIGURATION!
    for candidate_id in range(len(grid)):

        # Get parameters for this configuration
        parameters = grid[candidate_id]

        # Create empty list for this configuration's fold scores
        fold_scores = []

        # TRAIN AND VALIDATE ON ALL FIVE FOLDS!
        for fold in range(5):

            # Get training and validation row indices
            train_idx, valid_idx = cv_splits[fold]

            # Select the training data
            X_train = X.iloc[train_idx]
            y_train = y.iloc[train_idx]

            # Select the validation data
            X_validation = X.iloc[valid_idx]
            y_validation = y.iloc[valid_idx]

            # Create a fresh, untrained copy of the pipeline
            estimator = clone(template)

            # Apply the hyperparameters we want to test
            estimator.set_params(**parameters)

            # Fit preprocessing AND the model on training data
            estimator.fit(X_train, y_train)

            # Predict probabilities for validation titles
            probabilities = estimator.predict_proba(
                X_validation
            )[:, 1] #selects the probability of Netflix = 1

            # Calculate our four evaluation metrics
            metrics = score_predictions(
                y_validation, probabilities
            )

            # Save these metrics for averaging across folds
            fold_scores.append(metrics)

            # Save detailed results for this fold
            fold_rows.append({
                "Feature set": set_name,
                "Algorithm": model_name,
                "Candidate": candidate_id + 1,
                "Fold": fold + 1,
                **metrics
            })

        # SUMMARIZE THE FIVE FOLDS!!

        # Calculate average performance and log-loss SD
        summary = summarize_folds(fold_scores)

        # Save the overall results for this configuration
        candidate_rows.append({
            "Feature set": set_name,
            "Algorithm": model_name,
            "Candidate": candidate_id + 1,
            "Parameters": parameters,
            **summary
        })

        print("Finished setting", candidate_id + 1,
              "of", len(grid))

Total configurations: 28
Total model fits: 140

Feature set: Set 1
Algorithm: Logistic regression
Finished setting 1 of 4
Finished setting 2 of 4
Finished setting 3 of 4
Finished setting 4 of 4

Feature set: Set 1
Algorithm: Classification tree
Finished setting 1 of 9
Finished setting 2 of 9
Finished setting 3 of 9
Finished setting 4 of 9
Finished setting 5 of 9
Finished setting 6 of 9
Finished setting 7 of 9
Finished setting 8 of 9
Finished setting 9 of 9

Feature set: Set 1
Algorithm: Random forest
Finished setting 1 of 1

Feature set: Set 2
Algorithm: Logistic regression
Finished setting 1 of 4
Finished setting 2 of 4
Finished setting 3 of 4
Finished setting 4 of 4

Feature set: Set 2
Algorithm: Classification tree
Finished setting 1 of 9
Finished setting 2 of 9
Finished setting 3 of 9
Finished setting 4 of 9
Finished setting 5 of 9
Finished setting 6 of 9
Finished setting 7 of 9
Finished setting 8 of 9
Finished setting 9 of 9

Feature set: Set 2
Algorithm: Random forest
Finished se

**Comparison Table!!**

In [ ]:
# Reuse baseline results from above
baseline_summary = {
    "Log loss": baseline_results["Log loss"].mean(),
    "Log loss SD": baseline_results["Log loss"].std(),
    "Error (%)": baseline_results["Misclassification %"].mean(),
    "Precision (%)": baseline_results["Precision %"].mean(),
    "Recall (%)": baseline_results["Recall %"].mean()
}

# Copy best model results from previous cell
comparison = best_results.drop(
    columns=["Candidate", "Parameters"]
).copy()

# Add information about tuning settings
settings_tried = []
settings_selected = []
candidates_tried = []

for index, row in best_results.iterrows():
    model_name = row["Algorithm"]

    settings_tried.append(settings_text(param_grids[model_name]))
    settings_selected.append(settings_text(row["Parameters"]))
    candidates_tried.append(len(ParameterGrid(param_grids[model_name])))

comparison["Settings tried"] = settings_tried
comparison["Settings selected"] = settings_selected
comparison["Candidates tried"] = candidates_tried

# Add baseline results to comparison
baseline_row = {
    "Feature set": "None",
    "Algorithm": "Baseline",
    **baseline_summary,
    "Settings tried": "Training Netflix proportion",
    "Settings selected": "Training Netflix proportion",
    "Candidates tried": 1
}

comparison = pd.concat(
    [pd.DataFrame([baseline_row]), comparison],
    ignore_index=True
)

# Calculate improvement over baseline!
# Positive = better than the baseline (lower log loss)
comparison["Improvement vs baseline"] = (
    baseline_summary["Log loss"] - comparison["Log loss"]
)

column_order = [
    "Feature set", "Algorithm",
    "Log loss", "Log loss SD",
    "Improvement vs baseline",
    "Error (%)", "Precision (%)", "Recall (%)",
    "Candidates tried", "Settings tried", "Settings selected"
]
comparison = comparison[column_order]
display(comparison.round(6))

,Feature set,Algorithm,Log loss,Log loss SD,Improvement vs baseline,Error (%),Precision (%),Recall (%),Candidates tried,Settings tried,Settings selected
0,None,Baseline,0.662679,0.000059,0.000000,37.720475,NaN,0.000000,1,Training Netflix proportion,Training Netflix proportion
1,Set 1,Logistic regression,0.644774,0.003700,0.017905,38.006010,48.087793,7.770847,4,"C=[0.01, 0.1, 1, 10]",C=1
2,Set 1,Classification tree,0.625808,0.008839,0.036871,35.083108,55.395873,37.075097,9,"max_depth=[3, 5, 8]; min_samples_leaf=[25, 50,...",max_depth=8; min_samples_leaf=50
3,Set 1,Random forest,0.845658,0.051676,-0.182979,37.266930,50.763604,45.446661,1,Defaults; random_state=226,Defaults; random_state=226
4,Set 2,Logistic regression,0.632040,0.003914,0.030639,37.056839,52.283919,20.952927,4,"C=[0.01, 0.1, 1, 10]",C=1
5,Set 2,Classification tree,0.619075,0.005666,0.043604,35.486363,55.822087,31.929043,9,"max_depth=[3, 5, 8]; min_samples_leaf=[25, 50,...",max_depth=5; min_samples_leaf=25
6,Set 2,Random forest,0.689221,0.031949,-0.026542,32.613846,58.473030,46.670862,1,Defaults; random_state=226,Defaults; random_state=226


**COMPARISONS FOR REPORT & INTERPRETATION**

In [ ]:
# Find best model for each feature set
for set_name in ["Set 1", "Set 2"]:
    # Select results for this feature set
    group = best_results[best_results["Feature set"] == set_name]
    # Find model with the lowest log loss
    best = group.sort_values("Log loss").iloc[0]
    print(set_name, "best model:", best["Algorithm"])
    print("Log loss:", round(best["Log loss"], 6))

# Compare Set 1 and Set 2 for each algorithm
print("\nEffect of adding ratings and popularity:")

for algorithm in models:
    # Get results for this algorithm
    group = best_results[best_results["Algorithm"] == algorithm]
    # Get log loss for each feature set
    loss1 = group[group["Feature set"] == "Set 1"]["Log loss"].iloc[0]
    loss2 = group[group["Feature set"] == "Set 2"]["Log loss"].iloc[0]
    # Positive improvement means Set 2 performed better
    improvement = loss1 - loss2

    print(algorithm, "improvement:", round(improvement, 6))

Set 1 best model: Classification tree
Log loss: 0.625808
Set 2 best model: Classification tree
Log loss: 0.619075

Effect of adding ratings and popularity:
Logistic regression improvement: 0.012734
Classification tree improvement: 0.006733
Random forest improvement: 0.156437
